# 🧬 Phase 1: Biomedical KG — Data Download & RAPIDS GPU Preprocessing

**Project:** BioKG Disease Link Predictor  
**Dataset:** DRKG (Drug Repurposing Knowledge Graph) — 5.8M biological triples  
**Tech:** NVIDIA RAPIDS cuDF/cuGraph + BioBridge Alignment  

---

## ⚡ Before Starting
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells

## Step 0: Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ Stop! No GPU detected."
print(f"\n✅ GPU Connected: {torch.cuda.get_device_name(0)}")

## Step 1: Install NVIDIA RAPIDS
This takes ~3-5 minutes. ☕

In [ ]:
import torch
import subprocess
cuda_version = torch.version.cuda
major = int(cuda_version.split('.')[0]) if cuda_version else 12
rapids_suffix = 'cu11' if major == 11 else 'cu12'

print(f"📦 Installing RAPIDS for CUDA {cuda_version}...")
!pip install cudf-{rapids_suffix} cugraph-{rapids_suffix} --extra-index-url=https://pypi.nvidia.com -q
print("✅ RAPIDS installed.")

## Step 2: Download & Smart Extraction

In [ ]:
import requests
import tarfile
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR = Path('data/raw')
if DATA_DIR.exists():
    import shutil
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("📥 Downloading DRKG dataset...")
url = 'https://dgl-data.s3-accelerate.amazonaws.com/dataset/DRKG/drkg.tar.gz'
dest = DATA_DIR / 'drkg.tar.gz'

response = requests.get(url, stream=True)
total = int(response.headers.get('content-length', 0))
with open(dest, 'wb') as f, tqdm(total=total, unit='iB', unit_scale=True) as pb:
    for chunk in response.iter_content(8192):
        pb.update(f.write(chunk))

print("\n📦 Extracting...")
with tarfile.open(dest, 'r:gz') as tar:
    tar.extractall(path=DATA_DIR)

tsv_files = list(DATA_DIR.rglob('*.tsv'))
DRKG_TSV = max(tsv_files, key=lambda f: f.stat().st_size)
print(f"\n✅ Target File Found: {DRKG_TSV.name}")

## Step 3: GPU vs CPU Benchmark

In [ ]:
import time
import cudf
import pandas as pd

print('⚡ Starting Benchmark...')

t0 = time.time()
pdf = pd.read_csv(str(DRKG_TSV), sep='\t', header=None, names=['head', 'relation', 'tail'], encoding='latin-1')
cpu_time = time.time() - t0
print(f'🐢 pandas (CPU): {cpu_time:.2f}s')

t0 = time.time()
gdf = cudf.read_csv(str(DRKG_TSV), sep='\t', header=None, names=['head', 'relation', 'tail'])
gpu_time = time.time() - t0
print(f'⚡ cuDF (GPU):   {gpu_time:.2f}s')

print(f'\n🚀 Speedup: {cpu_time / max(gpu_time, 0.001):.1f}x!')

## Step 4: Save & Finalize

In [ ]:
PROC_DIR = Path('data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)
gdf.to_parquet(PROC_DIR / 'drkg.parquet')

print(f"✅ Processed data is ready in local Colab at {PROC_DIR}/")

# 📥 OPTIONAL: Download to your computer
from google.colab import files
print("\n💡 To download the processed Parquet file to your local computer, uncomment the line below:")
# files.download(str(PROC_DIR / 'drkg.parquet'))